# 01 — What Is Function Calling? — Practical Examples

**Covers**: FC protocol anatomy, first function call, message lifecycle, tool_choice modes

In [ ]:
import os, json
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. Your First Function Call — Step by Step

In [ ]:
# ── Define a simple tool ─────────────────────────────────────────────────
TOOLS = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get current weather for a city. Use when user asks about weather.',
        'parameters': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string', 'description': 'City name, e.g. Mumbai'},
                'units': {'type': 'string', 'enum': ['celsius', 'fahrenheit'], 'description': 'Temperature unit'}
            },
            'required': ['city']
        }
    }
}]

# ── Mock implementation ───────────────────────────────────────────────────
def get_weather(city: str, units: str = 'celsius') -> str:
    data = {
        'Mumbai':  {'celsius': 30, 'fahrenheit': 86,  'condition': 'Humid'},
        'London':  {'celsius': 12, 'fahrenheit': 54,  'condition': 'Cloudy'},
        'New York':{'celsius': 18, 'fahrenheit': 64,  'condition': 'Sunny'}
    }
    info = data.get(city, {'celsius': 22, 'fahrenheit': 72, 'condition': 'Clear'})
    temp = info[units]
    unit_sym = '°C' if units == 'celsius' else '°F'
    return f'{city}: {temp}{unit_sym}, {info["condition"]}'

print('Tools defined:', [t['function']['name'] for t in TOOLS])

In [ ]:
# ── Step 1: First LLM call — model decides to use tool ───────────────────
messages = [
    {'role': 'system', 'content': 'You are a helpful weather assistant.'},
    {'role': 'user',   'content': 'What is the weather in London?'}
]

response = client.chat.completions.create(
    model=MODEL, messages=messages,
    tools=TOOLS, tool_choice='auto'
)

msg = response.choices[0].message
print(f'finish_reason: {response.choices[0].finish_reason}')
print(f'content:       {msg.content}')
print(f'tool_calls:    {msg.tool_calls}')

In [ ]:
# ── Step 2: Inspect the tool call ────────────────────────────────────────
tc = msg.tool_calls[0]
print(f'Tool Call ID:   {tc.id}')
print(f'Tool Name:      {tc.function.name}')
print(f'Arguments JSON: {tc.function.arguments}')  # Note: it's a string!
args = json.loads(tc.function.arguments)            # Must parse it
print(f'Arguments dict: {args}')

In [ ]:
# ── Step 3: Execute tool + inject result ─────────────────────────────────
messages.append(msg)  # Add assistant message with tool_calls
result = get_weather(**args)
print(f'Tool result: {result}')

messages.append({
    'role': 'tool',
    'tool_call_id': tc.id,  # Must match
    'content': result
})

# ── Step 4: Final LLM call ───────────────────────────────────────────────
final_response = client.chat.completions.create(
    model=MODEL, messages=messages
)
print(f'\nFinal Answer: {final_response.choices[0].message.content}')

## 2. tool_choice Options

In [ ]:
user_msg = 'Tell me about London'
modes = [
    ('auto',     'auto'),
    ('none',     'none'),
    ('required', 'required'),
    ('forced',   {'type': 'function', 'function': {'name': 'get_weather'}})
]

for label, tc_val in modes:
    r = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': user_msg}],
        tools=TOOLS, tool_choice=tc_val, max_tokens=80
    )
    fr = r.choices[0].finish_reason
    has_tool = bool(r.choices[0].message.tool_calls)
    print(f'{label:<10} finish={fr:<12} tool_called={has_tool}')

## 3. Full Message Lifecycle Visualizer

In [ ]:
def run_full_loop(user_input: str) -> str:
    msgs = [
        {'role': 'system', 'content': 'Weather assistant. Use get_weather tool.'},
        {'role': 'user', 'content': user_input}
    ]
    print(f'MESSAGES AFTER STEP 0: {len(msgs)} messages')
    
    for step in range(4):
        r = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS)
        msg = r.choices[0].message
        msgs.append(msg)
        print(f'MESSAGES AFTER STEP {step+1}: {len(msgs)} messages | finish={r.choices[0].finish_reason}')
        
        if r.choices[0].finish_reason == 'tool_calls':
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                result = get_weather(**args)
                msgs.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
        else:
            return msg.content
    return 'max steps'

answer = run_full_loop('What is the weather in Mumbai and London?')
print(f'\nFINAL: {answer}')